# 실습 3. Boosting과 Stacking

## 데이터
- 분류: `sklearn.datasets.load_breast_cancer()`
- 회귀: `sklearn.datasets.load_diabetes()`

## 실습 목표
- GradientBoostingClassifier의 주요 하이퍼파라미터 비교
- HistGradientBoostingClassifier 학습과 평가
- StackingClassifier로 여러 모델의 예측 결합
- StackingRegressor로 회귀 모델 결합


In [5]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.ensemble import StackingClassifier, StackingRegressor
from sklearn.metrics import classification_report, root_mean_squared_error, r2_score

cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('분류 데이터:', X.shape, y.shape)
print('target names:', cancer.target_names)


분류 데이터: (569, 30) (569,)
target names: ['malignant' 'benign']


## 문제 1. GradientBoostingClassifier 기본 모델 학습

GradientBoostingClassifier를 학습하고 분류 성능을 확인하세요.

### 요구사항
- `n_estimators=100`
- `learning_rate=0.1`
- `max_depth=3`
- `random_state=42`
- 학습셋/평가셋 accuracy와 `classification_report()` 출력

### 실행 결과
```text
학습셋 accuracy: 약 1.0000
평가셋 accuracy: 약 0.9474 이상
classification_report가 출력됨
```


In [2]:
gbc = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gbc.fit(X_train, y_train)
y_pred = gbc.predict(X_test)

print("학습셋 accuracy: ", gbc.score(X_train, y_train))
print("평가셋 accuracy: ", gbc.score(X_test, y_test))
print("classification_report: \n", classification_report(y_test, y_pred))

학습셋 accuracy:  1.0
평가셋 accuracy:  0.956140350877193
classification_report: 
               precision    recall  f1-score   support

           0       0.97      0.90      0.94        42
           1       0.95      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



## 문제 2. learning_rate와 n_estimators 비교

learning_rate와 n_estimators 조합별 점수를 비교하세요.

### 요구사항
- `learning_rate`: `[0.03, 0.1, 0.3]`
- `n_estimators`: `[50, 100, 200]`
- 결과 컬럼: `learning_rate`, `n_estimators`, `train_accuracy`, `test_accuracy`
- `test_accuracy` 기준 내림차순 정렬

### 실행 결과
```text
9개 조합의 결과 표가 출력됨
상위 조합은 test_accuracy가 약 0.95 이상으로 출력됨
```


In [8]:
learning_rate = [0.03, 0.1, 0.3]
n_estimators = [50, 100, 200]

bs = GradientBoostingClassifier(
    max_depth=3,
    random_state=42,
)

gs = GridSearchCV(
    estimator=bs,
    param_grid=dict(learning_rate=learning_rate, n_estimators=n_estimators),
    scoring='accuracy',
    cv=4,
    n_jobs=4,
)

gs.fit(X_train, y_train)

gs_result = pd.DataFrame(gs.cv_results_)
gs_result

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_learning_rate,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,mean_test_score,std_test_score,rank_test_score
0,0.112396,0.003477,0.002253,0.000435,0.03,50,"{'learning_rate': 0.03, 'n_estimators': 50}",0.973684,0.921053,0.929825,0.973451,0.949503,0.024264,8
1,0.238315,0.003546,0.002386,0.000429,0.03,100,"{'learning_rate': 0.03, 'n_estimators': 100}",0.956140,0.912281,0.938596,0.973451,0.945117,0.022611,9
2,0.521773,0.054555,0.002386,0.000421,0.03,200,"{'learning_rate': 0.03, 'n_estimators': 200}",0.956140,0.929825,0.956140,0.964602,0.951677,0.013081,6
3,0.112960,0.005707,0.002700,0.000554,0.10,50,"{'learning_rate': 0.1, 'n_estimators': 50}",0.956140,0.938596,0.947368,0.964602,0.951677,0.009704,6
4,0.236884,0.011985,0.002546,0.000487,0.10,100,"{'learning_rate': 0.1, 'n_estimators': 100}",0.956140,0.947368,0.973684,0.973451,0.962661,0.011339,3
5,0.518039,0.048252,0.002638,0.000420,0.10,200,"{'learning_rate': 0.1, 'n_estimators': 200}",0.956140,0.947368,0.964912,0.973451,0.960468,0.009729,4
6,0.149075,0.028728,0.003516,0.001227,0.30,50,"{'learning_rate': 0.3, 'n_estimators': 50}",0.964912,0.938596,0.947368,0.982301,0.958295,0.016789,5
7,0.231645,0.005196,0.002931,0.000631,0.30,100,"{'learning_rate': 0.3, 'n_estimators': 100}",0.964912,0.947368,0.964912,0.991150,0.967086,0.015631,1
8,0.367878,0.020938,0.002851,0.000887,0.30,200,"{'learning_rate': 0.3, 'n_estimators': 200}",0.956140,0.938596,0.964912,0.991150,0.962700,0.018963,2


## 문제 3. HistGradientBoostingClassifier 학습

HistGradientBoostingClassifier를 학습하고 평가셋 accuracy를 확인하세요.

### 요구사항
- `max_iter=100`
- `learning_rate=0.1`
- `max_leaf_nodes=15`
- `random_state=42`
- 학습셋/평가셋 accuracy 출력

### 실행 결과
```text
학습셋 accuracy와 평가셋 accuracy가 출력됨
평가셋 accuracy는 약 0.94 이상으로 출력됨
```


In [9]:
hgbc = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=15,
    random_state=42
)

hgbc.fit(X_train, y_train)

y_pred = hgbc.predict(X_test)

print("학습셋: ", hgbc.score(X_train, y_train))
print("평가셋: ", hgbc.score(X_test, y_test))


학습셋:  1.0
평가셋:  0.956140350877193


## 문제 4. StackingClassifier 구성과 비교

Logistic Regression, KNN, Decision Tree를 기본 모델로 사용해 StackingClassifier를 구성하세요.

### 요구사항
- Logistic Regression과 KNN은 `Pipeline([('scaler', StandardScaler()), ...])`로 구성
- Decision Tree는 `max_depth=4`, `random_state=42`
- final estimator는 `LogisticRegression(max_iter=1000)` 사용
- 개별 모델과 Stacking 모델의 train/test accuracy를 표로 비교

### 실행 결과
```text
logistic_regression, knn, decision_tree, stacking 4개 모델의 점수 표가 출력됨
```


In [10]:
base_classifiers = [
    (
        'logistic_regression',
        Pipeline([
            # LogisticRegression은 feature 스케일 영향을 받을 수 있으므로 Pipeline 안에서 스케일링함.
            # Pipeline을 사용하면 cross validation 과정에서도 train fold에만 scaler가 fit되어 데이터 누수를 막을 수 있음.
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(max_iter=1000))
        ])
    ),
    (
        'knn',
        Pipeline([
            # KNN은 거리 기반 모델이므로 feature 단위가 다르면 큰 숫자 feature가 판단을 지배할 수 있음.
            ('scaler', StandardScaler()),
            ('model', KNeighborsClassifier(n_neighbors=5))
        ])
    ),
    (
        'decision_tree',
        # DecisionTree는 feature의 대소 비교로 분기하므로 스케일링이 필수는 아님.
        # max_depth=4로 트리 복잡도를 제한해 과대적합을 줄임.
        DecisionTreeClassifier(max_depth=4, random_state=42)
    )
]

stacking_clf = StackingClassifier(
    estimators=base_classifiers,
    final_estimator=LogisticRegression(max_iter=1000),
    cv= 5, # 교차 검증 5회
    n_jobs= 1
)

stacking_clf.fit(X_train, y_train)

stacking_clf.score(X_train, y_train)

0.9912087912087912

## 문제 5. StackingRegressor 회귀 모델 학습

Diabetes 데이터로 StackingRegressor를 만들고 회귀 지표를 비교하세요.

### 요구사항
- 기본 모델: Ridge, KNeighborsRegressor, DecisionTreeRegressor
- Ridge와 KNN은 스케일링 Pipeline 사용
- final estimator는 `Ridge()` 사용
- 평가셋 R2와 RMSE를 표로 출력

### 실행 결과
```text
ridge, knn_regressor, decision_tree_regressor, stacking_regressor 4개 모델의 회귀 지표가 출력됨
```


In [ ]:
diabetes = load_diabetes(as_frame=True)
reg_X = diabetes.data
reg_y = diabetes.target

reg_X_train, reg_X_test, reg_y_train, reg_y_test = train_test_split(
    reg_X,
    reg_y,
    test_size=0.2,
    random_state=42
)